# Importing and vizualizing dataset

In [1]:
import torch
import torch.nn as nn
import torchvision
import pandas as pd
import torchvision.transforms as transforms

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from pathlib import Path

In [2]:
filedir = Path('./datasets/')

train_dataframe = pd.read_csv(filedir / 'train_data.csv')
test_dataframe = pd.read_csv(filedir / 'test_data.csv')

In [3]:
train_dataframe.head()

,ID,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety
0,Calc-Training_P_00005_RIGHT_CC_1,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
1,Calc-Training_P_00005_RIGHT_MLO_1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
2,Calc-Training_P_00007_LEFT_CC_1,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
3,Calc-Training_P_00007_LEFT_MLO_1,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
4,Calc-Training_P_00008_LEFT_CC_1,P_00008,1,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3


# Data preprocessing

## 2.1. Dataset loader creation

In [4]:
class BreastCancerDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = f"{self.image_dir}/{row['ID']}.jpg"
        image = Image.open(image_path).convert("L")  # Set L if greyscale, RGB otherwise
        
        label = row['pathology']  # Target label
        label = 0 if label == 'BENIGN_WITHOUT_CALLBACK' else 1 if label == 'BENIGN' else 2  # Encoding classes
        
        if self.transform:
            image = self.transform(image)

        return image, label

## 2.2. Image transformation and scaling

In [5]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Resize every image to 128 X 128 matrix
    transforms.ToTensor()  # Default normalization to [0, 1]
])

In [6]:
train_dataset = BreastCancerDataset(train_dataframe, filedir / 'train_cropped_images/', transform=transform)
test_dataset = BreastCancerDataset(test_dataframe, filedir / 'test_cropped_images/', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# Neural network model design

## 3.1. Convolutional Neural Networks 

In [7]:
class CNN(nn.Module):
    def __init__(self, num_classes, additional_features_size=0):
        super(CNN, self).__init__()
        # Convolutional layers
        self.conv_layer1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.batch_norm1 = nn.BatchNorm2d(32)
        self.conv_layer2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.batch_norm2 = nn.BatchNorm2d(64)
        self.max_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.dropout1 = nn.Dropout(0.8)
        
        self.conv_layer3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.batch_norm3 = nn.BatchNorm2d(128)
        self.conv_layer4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.batch_norm4 = nn.BatchNorm2d(128)
        self.max_pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.dropout2 = nn.Dropout(0.4)
        # Fully connected layers
        self.flattened_size = None  # To be calculated dynamically
        self.fc = None  # To be initialized dynamically
        self.relu = nn.LeakyReLU(0.1)  # Change to leakyReLu
        # Final classification layer
        self.fc_final = nn.Linear(64, num_classes)
        
    def forward(self, x, additional_features=None):
        # Convolutional layers with ReLU activation
        x = self.relu(self.conv_layer1(x))
        x = self.relu(self.batch_norm1(x))
        x = self.relu(self.conv_layer2(x))
        x = self.relu(self.batch_norm2(x))
        x = self.max_pool1(x)

        x = self.dropout1(x)
        
        x = self.relu(self.conv_layer3(x))
        x = self.relu(self.batch_norm3(x))
        x = self.relu(self.conv_layer4(x))
        x = self.relu(self.batch_norm4(x))
        x = self.max_pool2(x)

        x = self.dropout2(x)
        # Flatten dynamically
        x = x.view(x.size(0), -1)
        
        # Initialize fc1 if not done (dynamic size)
        if self.fc is None:
            self.flattened_size = x.size(1)
            self.fc = nn.Linear(self.flattened_size, 64)
            self.fc.to(x.device)
        
        x = self.relu(self.fc(x))
        # Final classification layer
        x = self.fc_final(x)
        return x

## 3.2. Hyperparameter definition

In [10]:
num_classes = len(train_dataframe['pathology'].unique())  # There are 3 target classes

num_epochs = 25  # Training period

model = CNN(num_classes)  # Set model parameters

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5)  # Change learning rate if stagnate for 5 epochs

criterion = nn.CrossEntropyLoss()  # Cross entropy loss between input logits and target

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Whether to train on GPU (cuda) or CPU

model.to(device)

CNN(
  (conv_layer1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv_layer2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (max_pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.8, inplace=False)
  (conv_layer3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv_layer4): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (max_pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout2): Dropout(p=0.4, inplace=False)
  (

## 3.3. Training loop

In [11]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    all_preds = []
    all_labels = []

    # Training loop
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate loss and predictions
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}", end=", ")
    
    # Validation loop
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(test_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    scheduler.step(val_loss)
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")


Epoch 1/25, Loss: 1.1429, Accuracy: 0.3472, Validation Loss: 1.1013, Accuracy: 0.2546
Epoch 2/25, Loss: 1.0693, Accuracy: 0.3977, Validation Loss: 1.0577, Accuracy: 0.4417
Epoch 3/25, Loss: 1.0501, Accuracy: 0.4320, Validation Loss: 1.0393, Accuracy: 0.4509
Epoch 4/25, Loss: 1.0128, Accuracy: 0.4722, Validation Loss: 1.0316, Accuracy: 0.4540
Epoch 5/25, Loss: 1.0185, Accuracy: 0.4799, Validation Loss: 1.0286, Accuracy: 0.4632
Epoch 6/25, Loss: 1.0063, Accuracy: 0.4825, Validation Loss: 1.0177, Accuracy: 0.4755
Epoch 7/25, Loss: 1.0051, Accuracy: 0.4812, Validation Loss: 1.0121, Accuracy: 0.4877
Epoch 8/25, Loss: 0.9875, Accuracy: 0.4948, Validation Loss: 1.0180, Accuracy: 0.4877
Epoch 9/25, Loss: 0.9856, Accuracy: 0.4890, Validation Loss: 0.9979, Accuracy: 0.4724
Epoch 10/25, Loss: 0.9774, Accuracy: 0.5039, Validation Loss: 1.0125, Accuracy: 0.4632
Epoch 11/25, Loss: 0.9865, Accuracy: 0.4754, Validation Loss: 0.9966, Accuracy: 0.4632
Epoch 12/25, Loss: 0.9803, Accuracy: 0.4935, Validat

In [ ]:
# save_path = Path("./model")
# torch.save(model.state_dict(), save_path)

## 3.4. Using pretrained models

Considering custom made convolutional neural network model yields unsatisfactory results, can pretrained models offer better performance?

In [18]:
model = torchvision.models.resnet34(weights="IMAGENET1K_V1")

model.conv1 = nn.Conv2d(1, 64, kernel_size=7, padding=1, bias=False)

num_ftrs = model.fc.in_features  # Get the number of input features for the final layer
model.fc = nn.Linear(num_ftrs, 3)  # Replace with a new layer for 3 classes

model.to(device)

ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Since we're performing fine tuning changing all parameters is not necesarry.

In [19]:
for param in model.parameters():
    param.requires_grad = False  # Freeze all layers

for param in model.fc.parameters():
    param.requires_grad = True  # Unfreeze the last fully connected layer

ResNet takes 224 x 224 input so scaling images to specific dimesions is required.

In [25]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485], std=[0.229])  # ResNet normalization
])

train_dataset = BreastCancerDataset(train_dataframe, filedir / 'train_cropped_images/', transform=transform)
test_dataset = BreastCancerDataset(test_dataframe, filedir / 'test_cropped_images/', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [21]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)

In [26]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    all_preds = []
    all_labels = []

    # Training loop
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate loss and predictions
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}", end=", ")
    
    # Validation loop
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(test_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    scheduler.step(val_loss)
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

Epoch 1/25, Loss: 1.0423, Accuracy: 0.4573, Validation Loss: 1.0425, Accuracy: 0.4233
Epoch 2/25, Loss: 0.9385, Accuracy: 0.5408, Validation Loss: 0.9588, Accuracy: 0.5276
Epoch 3/25, Loss: 0.8957, Accuracy: 0.5648, Validation Loss: 0.9554, Accuracy: 0.5460
Epoch 4/25, Loss: 0.8781, Accuracy: 0.5816, Validation Loss: 0.8952, Accuracy: 0.5460
Epoch 5/25, Loss: 0.8631, Accuracy: 0.5907, Validation Loss: 0.9891, Accuracy: 0.4969
Epoch 6/25, Loss: 0.8489, Accuracy: 0.5842, Validation Loss: 0.9536, Accuracy: 0.5092
Epoch 7/25, Loss: 0.8502, Accuracy: 0.5764, Validation Loss: 0.8772, Accuracy: 0.5307
Epoch 8/25, Loss: 0.8157, Accuracy: 0.6159, Validation Loss: 0.9325, Accuracy: 0.5399
Epoch 9/25, Loss: 0.8126, Accuracy: 0.6211, Validation Loss: 0.8776, Accuracy: 0.5337
Epoch 10/25, Loss: 0.8273, Accuracy: 0.6140, Validation Loss: 0.9753, Accuracy: 0.5276
Epoch 11/25, Loss: 0.8080, Accuracy: 0.6185, Validation Loss: 0.8604, Accuracy: 0.5399
Epoch 12/25, Loss: 0.7956, Accuracy: 0.6418, Validat